<a href="https://colab.research.google.com/github/astiakamila/Processing-Text/blob/main/102_(Velyn)_162_(Astia)_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1> Tugas Pemteks Web Scarpping </h1>
<h2> Astia Kamila Dwi Septiani </h2>
<h3> NIM: 25031554162 </h3>
<h2> Zikrah Neisya Fabiola Evelyna </h2>
<h3> Nim: 25031554102 </h3>

<b> <h2>Web sraping with HTTP</h2> </b>

Install dan import library

In [ ]:
!pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO

Menentukan URL dan melakukan HTTP Request

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print("HASIL HTTP REQUEST")
print("URL         :", url)
print("Status Code :", response.status_code)
print("Panjang HTML:", len(response.text))

HASIL HTTP REQUEST
URL         : https://en.wikipedia.org/wiki/List_of_highest-grossing_films
Status Code : 200
Panjang HTML: 2203349


Mengecek apakah request berhasil

In [ ]:
if response.status_code == 200:
    print("Status: Berhasil mengambil halaman website")
else:
    print("Status: Gagal mengambil halaman website")

Status: Berhasil mengambil halaman website


Membaca HTML dan mengambil tabel

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

table = soup.find("table", class_="wikitable")

df = pd.read_html(StringIO(str(table)))[0]

print("Jumlah baris :", len(df))
print("Jumlah kolom :", len(df.columns))
print("\nNama kolom:")
print(df.columns.tolist())

Jumlah baris : 50
Jumlah kolom : 6

Nama kolom:
['Rank', 'Peak', 'Title', 'Worldwide gross', 'Year', 'Ref']


Memilih kolom yang dibutuhkan

In [ ]:
df_film = df[["Rank", "Title", "Worldwide gross", "Year"]].copy()

df_film.head()

,Rank,Title,Worldwide gross,Year
0,1,Avatar,"$2,923,710,708",2009
1,2,Avengers: Endgame †,"$2,808,701,328",2019
2,3,Spider-Man: Brand New Day †,"$2,481,707,775",2026
3,4,Avatar: The Way of Water,"$2,334,484,620",2022
4,5,Titanic,"T$2,257,906,828",1997


Membersihkan nama film

In [ ]:
df_film.loc[:, "Title"] = (
    df_film["Title"]
    .astype(str)
    .str.replace("†", "", regex=False)
    .str.strip()
)

Membersihkan data pendapatan film

In [ ]:
df_film["Worldwide gross"] = (
    df_film["Worldwide gross"]
    .astype(str)
    .str.replace(r"[^0-9]", "", regex=True)
)

df_film["Worldwide gross"] = pd.to_numeric(
    df_film["Worldwide gross"],
    errors="coerce"
)

print(df_film["Worldwide gross"].head())

0    2923710708
1    2808701328
2    2481707775
3    2334484620
4    2257906828
Name: Worldwide gross, dtype: int64


Mengubah tahun menjadi angka

In [ ]:
df_film.loc[:, "Year"] = pd.to_numeric(
    df_film["Year"],
    errors="coerce"
)

Menampilkan 10 film teratas

In [ ]:
print("10 FILM TERLARIS")

print(df_film.head(10).to_string(index=False))

10 FILM TERLARIS
Rank                        Title  Worldwide gross  Year
   1                       Avatar       2923710708  2009
   2            Avengers: Endgame       2808701328  2019
   3    Spider-Man: Brand New Day       2481707775  2026
   4     Avatar: The Way of Water       2334484620  2022
   5                      Titanic       2257906828  1997
   6                     Ne Zha 2       2217758382  2025
   7 Star Wars: The Force Awakens       2068223624  2015
   8       Avengers: Infinity War       2048359754  2018
   9      Spider-Man: No Way Home       1922598800  2021
  10                   Zootopia 2       1866647950  2025


<b> <h2> Web sraping with API </h2> </b>

Library

In [ ]:
import requests
import pandas as pd

Input nama kota

In [ ]:
kota = input("Masukkan nama kota: ")

print("Kota yang dipilih:", kota)

Masukkan nama kota: bekasi
Kota yang dipilih: bekasi


Mencari koordinat kota

In [ ]:
geo_url = "https://geocoding-api.open-meteo.com/v1/search"

geo_params = {
    "name": kota,
    "count": 1,
    "language": "id",
    "format": "json"
}

geo_response = requests.get(geo_url, params=geo_params)

print("Status Code:", geo_response.status_code)

Status Code: 200


Mengambil latitude dan longitude

In [ ]:
geo_data = geo_response.json()

if "results" in geo_data:
    lokasi = geo_data["results"][0]

    nama_kota = lokasi["name"]
    latitude = lokasi["latitude"]
    longitude = lokasi["longitude"]

    print("Kota      :", nama_kota)
    print("Latitude  :", latitude)
    print("Longitude :", longitude)
else:
    print("Kota tidak ditemukan.")

Kota      : Kota Bekasi
Latitude  : -6.2349
Longitude : 106.9896


Mengambil data cuaca dari API

In [ ]:
weather_url = "https://api.open-meteo.com/v1/forecast"

weather_params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "timezone": "auto",
    "forecast_days": 1
}

weather_response = requests.get(
    weather_url,
    params=weather_params
)

print("Status Code:", weather_response.status_code)

Status Code: 200


Mengubah data JSON menjadi DataFrame

In [ ]:
weather_data = weather_response.json()

hourly = weather_data["hourly"]

df_cuaca = pd.DataFrame({
    "Waktu": hourly["time"],
    "Suhu (°C)": hourly["temperature_2m"],
    "Kelembapan (%)": hourly["relative_humidity_2m"],
    "Kecepatan Angin (km/jam)": hourly["wind_speed_10m"]
})

df_cuaca.head()

,Waktu,Suhu (°C),Kelembapan (%),Kecepatan Angin (km/jam)
0,2026-09-18T00:00,27.8,72,4.6
1,2026-09-18T01:00,27.6,72,4.2
2,2026-09-18T02:00,27.0,77,3.2
3,2026-09-18T03:00,26.7,78,3.4
4,2026-09-18T04:00,26.4,80,3.2


Melihat seluruh data

In [ ]:
print("DATA CUACA DI", nama_kota.upper())

print(df_cuaca.to_string(index=False))

DATA CUACA DI KOTA BEKASI
           Waktu  Suhu (°C)  Kelembapan (%)  Kecepatan Angin (km/jam)
2026-09-18T00:00       27.8              72                       4.6
2026-09-18T01:00       27.6              72                       4.2
2026-09-18T02:00       27.0              77                       3.2
2026-09-18T03:00       26.7              78                       3.4
2026-09-18T04:00       26.4              80                       3.2
2026-09-18T05:00       26.2              81                       4.1
2026-09-18T06:00       25.9              83                       2.5
2026-09-18T07:00       27.6              73                       6.9
2026-09-18T08:00       28.8              68                       8.9
2026-09-18T09:00       30.1              62                       9.5
2026-09-18T10:00       31.8              54                       8.0
2026-09-18T11:00       33.4              47                       6.4
2026-09-18T12:00       34.5              43                     

Statistik sederhana

In [ ]:
print("STATISTIK CUACA")

print("Suhu minimum :", df_cuaca["Suhu (°C)"].min(), "°C")
print("Suhu maksimum:", df_cuaca["Suhu (°C)"].max(), "°C")
print("Suhu rata-rata:", round(df_cuaca["Suhu (°C)"].mean(), 2), "°C")

STATISTIK CUACA
Suhu minimum : 25.9 °C
Suhu maksimum: 35.3 °C
Suhu rata-rata: 29.15 °C


Menyimpan hasil ke CSV

In [ ]:
nama_file = "data_cuaca_" + nama_kota.lower() + ".csv"

df_cuaca.to_csv(nama_file, index=False)

print("Data berhasil disimpan sebagai:", nama_file)

Data berhasil disimpan sebagai: data_cuaca_kota bekasi.csv
